# Vicon IK → hip moment vs (Vicon ID + applied)

Pipeline:

1. **Vicon IK → TCN** on the mocap/ID clock (cascade input LPF matching deploy).
2. **Xcorr-align** that prediction to **Vicon ID / mass** (find lag, shift pred).
3. **Separately** map telemetry **applied torque** onto the ID clock via GPIO offset (from `compare_processed_hip_exo_id` cache) — not via the pred↔ID xcorr.
4. Compare aligned pred vs **(Vicon ID + applied) / mass**.

- Checkpoint: `runs/0512_ik_id_hip_offline_zero_phase/best_model.pt`
- Pred metric LPF: zero-phase 6 Hz; ID/applied in GT: causal 6 Hz (hip_exo style)
- Trim: 10 s each end after alignment
- Trials: same list as `compare_processed_hip_exo_id`

Set `FORCE_REPLAY = True` to re-run; `False` loads the metrics CSV.


In [ ]:
import inspect
import io
import re
import sys
from pathlib import Path
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from scipy.interpolate import splrep, splev
from scipy.signal import butter, sosfilt, sosfiltfilt

PROJECT_ROOT = Path('/home/metamobility3/Jinwoo/os_kinetics').resolve()
sys.path.insert(0, str(PROJECT_ROOT))
from model import TCN

PROCESSED_ROOT = Path('/media/metamobility3/Samsung_T52/Results/processed')
CACHE_PATH = PROJECT_ROOT / 'analysis/cache/compare_processed_hip_exo_id.npz'
HIP_CKPT = PROJECT_ROOT / 'runs/0512_ik_id_hip_offline_zero_phase/best_model.pt'
OUT_DIR = PROJECT_ROOT / 'analysis/paper_outputs/kinematic_input_quality'
FIG_DIR = OUT_DIR / 'figures' / 'hip_vicon_ik_vs_id'
METRICS_CSV = PROJECT_ROOT / 'analysis/cache/vicon_vs_hip_exo_vicon_ik_vs_id.csv'
DEPLOY_METRICS_CSV = PROJECT_ROOT / 'analysis/cache/compare_processed_hip_exo_id_metrics.csv'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
FORCE_REPLAY = False

EXO_KIND = 'hip-exo'
JOINT_R, JOINT_L = 'hip_flexion_r', 'hip_flexion_l'
MOMENT_COL = 'hip_flexion_r_moment'
TRIM_START_SEC = TRIM_END_SEC = 10.0
LPF_CUTOFF_HZ, LPF_ORDER = 6.0, 4
ANGLE_LPF_HZ, VEL_LPF_HZ = 6.0, 15.0
XCORR_MAX_LAG_SEC = 2.0
XCORR_SKIP_SEC = 5.0

SUBJECT_TOKEN_TO_DIR = {
    'ab01_jinwoo': 'AB01_Jinwoo', 'ab02_oscar': 'AB02_Oscar', 'ab03_ilseung': 'AB03_Ilseung',
    'ab04_changseob': 'AB04_Changseob', 'ab05_maria': 'AB05_Maria', 'ab06_jimin': 'AB06_Jimin',
    'ab07_amy': 'AB07_Amy', 'ab08_seokhyun': 'AB08_Seokhyun',
}
SUBJECT_MASS_KG = {
    'ab01_jinwoo': 88.0, 'ab02_oscar': 71.1, 'ab03_ilseung': 84.4, 'ab04_changseob': 74.0,
    'ab05_maria': 55.0, 'ab06_jimin': 82.6, 'ab07_amy': 51.3, 'ab08_seokhyun': 71.9,
}

WAVES: Dict[str, Dict] = {}


class CausalLowPass:
    """Match hip-exo-ctrl-V2 cascade._CausalLowPass."""

    def __init__(self, fs_hz: float, cutoff_hz: float, order: int = 4):
        self.order = max(1, int(order))
        if cutoff_hz <= 0.0:
            self.alpha = 1.0
        else:
            dt = 1.0 / float(fs_hz)
            tau = 1.0 / (2.0 * np.pi * float(cutoff_hz))
            self.alpha = dt / (tau + dt)
        self.state = [0.0] * self.order
        self.initialized = False

    def update(self, x: float) -> float:
        x = float(x)
        if not self.initialized:
            self.state = [x] * self.order
            self.initialized = True
            return x
        y = x
        for i in range(self.order):
            self.state[i] = self.state[i] + self.alpha * (y - self.state[i])
            y = self.state[i]
        return float(y)


def apply_cascade_lpf(x, fs_hz, cutoff_hz, order=4):
    filt = CausalLowPass(fs_hz, cutoff_hz, order)
    out = np.empty(len(x), dtype=np.float64)
    for i, v in enumerate(np.asarray(x, dtype=np.float64)):
        out[i] = filt.update(v if np.isfinite(v) else 0.0)
    return out


def read_sto(path: Path):
    with open(path) as f:
        lines = f.readlines()
    end_idx = next(i for i, l in enumerate(lines) if l.strip().lower() == 'endheader')
    cols = lines[end_idx + 1].strip().split()
    data = np.loadtxt(io.StringIO(''.join(lines[end_idx + 2:])))
    if data.ndim == 1:
        data = data.reshape(1, -1)
    return cols, data


def fill_nan(arr):
    arr = np.asarray(arr, dtype=np.float64).copy()
    finite = np.isfinite(arr)
    if finite.all() or not finite.any():
        return arr
    arr[~finite] = np.interp(np.flatnonzero(~finite), np.flatnonzero(finite), arr[finite])
    return arr


def zp_lpf(x, fs_hz, cutoff_hz=LPF_CUTOFF_HZ, order=LPF_ORDER):
    arr = fill_nan(x)
    nyq = 0.5 * float(fs_hz)
    if cutoff_hz <= 0 or cutoff_hz >= nyq:
        return arr
    sos = butter(int(order), float(cutoff_hz) / nyq, btype='low', output='sos')
    return sosfiltfilt(sos, arr)


def causal_butter(x, fs_hz, cutoff_hz=LPF_CUTOFF_HZ, order=LPF_ORDER):
    arr = fill_nan(x)
    nyq = 0.5 * float(fs_hz)
    sos = butter(int(order), float(cutoff_hz) / nyq, btype='low', output='sos')
    return sosfilt(sos, arr)


def rmse_r2(y_true, y_pred):
    m = np.isfinite(y_true) & np.isfinite(y_pred)
    if m.sum() < 2:
        return np.nan, np.nan
    e = y_pred[m] - y_true[m]
    rmse = float(np.sqrt(np.mean(e ** 2)))
    ss_tot = float(np.sum((y_true[m] - np.mean(y_true[m])) ** 2))
    return rmse, float(1.0 - np.sum(e ** 2) / (ss_tot + 1e-12))


def infer_fs(time_s, default=100.0):
    dt = np.diff(np.asarray(time_s, dtype=np.float64))
    dt = dt[np.isfinite(dt) & (dt > 0)]
    return float(1.0 / np.median(dt)) if dt.size else float(default)


def subject_token(stem: str) -> str:
    return '_'.join(stem.lower().split('_')[:2])


def trial_cond_speed(stem: str):
    parts = stem.lstrip('_').lower().split('_')
    cond = re.sub(r'\d+$', '', parts[4]).upper()
    return cond, parts[3]


def resolve_paths(stem: str) -> Dict[str, Path]:
    cond, speed = trial_cond_speed(stem)
    subj = PROCESSED_ROOT / SUBJECT_TOKEN_TO_DIR[subject_token(stem)]
    stem = stem.lstrip('_')
    telem = None
    for name in (f'{stem}.npz', f'_{stem}.npz'):
        p = PROJECT_ROOT / name
        if p.is_file():
            telem = p
            break
    if telem is None:
        raise FileNotFoundError(stem)
    return {
        'npz': telem,
        'ik': subj / EXO_KIND / 'ik' / f'{cond}_{speed}_ik.mot',
        'id': subj / EXO_KIND / 'id' / f'{cond}_{speed}_id.sto',
    }


def load_gpio_offsets() -> Dict[str, float]:
    data = np.load(str(CACHE_PATH), allow_pickle=True)
    out = {}
    for key in data['trial_keys']:
        p = str(key)
        out[p] = float(data[f'{p}__meta'][3])
    return out


def spline_vel(t, ang):
    t = np.asarray(t, dtype=np.float64)
    ang = np.asarray(ang, dtype=np.float64)
    if len(t) < 5:
        fs = infer_fs(t)
        vel = np.zeros_like(ang)
        vel[1:] = np.diff(ang) * fs
        return vel
    return np.asarray(splev(t, splrep(t, ang, s=0, k=3), der=1), dtype=np.float64)


def _zscore(x):
    x = np.asarray(x, dtype=np.float64)
    m = np.isfinite(x)
    out = np.zeros_like(x)
    if m.sum() < 2:
        return out
    mu, sd = float(np.mean(x[m])), float(np.std(x[m]))
    if sd < 1e-12:
        return out
    out[m] = (x[m] - mu) / sd
    return out


def shift_samples(x, lag: int):
    """Positive lag: delay x (x appears later). lag = argmax corr(pred_shifted, ref) with pred delayed."""
    x = np.asarray(x, dtype=np.float64)
    out = np.full_like(x, np.nan)
    lag = int(lag)
    if lag == 0:
        return x.copy()
    if lag > 0:
        out[lag:] = x[: len(x) - lag]
    else:
        out[:lag] = x[-lag:]
    return out


def xcorr_lag_samples(moving, fixed, fs_hz, max_lag_s=XCORR_MAX_LAG_SEC, skip_s=XCORR_SKIP_SEC):
    """
    Lag such that shift_samples(moving, lag) best matches fixed.
    Positive lag => moving leads fixed (delay moving to align).
    """
    moving = np.asarray(moving, dtype=np.float64)
    fixed = np.asarray(fixed, dtype=np.float64)
    n = min(len(moving), len(fixed))
    moving, fixed = moving[:n], fixed[:n]
    max_lag = int(round(max_lag_s * fs_hz))
    skip = int(round(skip_s * fs_hz))
    best_lag, best_score = 0, -np.inf
    for lag in range(-max_lag, max_lag + 1):
        shifted = shift_samples(moving, lag)
        m = np.isfinite(shifted) & np.isfinite(fixed)
        m[:skip] = False
        if m.sum() < int(round(2.0 * fs_hz)):
            continue
        score = float(np.dot(_zscore(shifted)[m], _zscore(fixed)[m]) / m.sum())
        if score > best_score:
            best_score, best_lag = score, lag
    return int(best_lag), float(best_score)


def _tcn_kwargs(cfg):
    allowed = {k for k in inspect.signature(TCN.__init__).parameters if k != 'self'}
    return {k: v for k, v in cfg.items() if k in allowed}


@torch.no_grad()
def run_bilateral(model, ang_r, vel_r, ang_l, vel_l, window):
    n = min(len(ang_r), len(vel_r), len(ang_l), len(vel_l))
    pred = np.full(n, np.nan, dtype=np.float64)
    for t_end in range(window - 1, n):
        i0 = t_end - window + 1
        x_r = np.stack([ang_r[i0:t_end + 1], vel_r[i0:t_end + 1]], 0).astype(np.float32)
        x_l = np.stack([ang_l[i0:t_end + 1], vel_l[i0:t_end + 1]], 0).astype(np.float32)
        xt = torch.from_numpy(np.stack([x_r, x_l], 0)).to(DEVICE)
        pred[t_end] = float(model(xt)[0, 0, -1].cpu())
    return pred


def replay_one(stem: str, gpio_offset_s: float, model, window: int) -> Dict:
    paths = resolve_paths(stem)
    mass = SUBJECT_MASS_KG[subject_token(stem)]

    # --- Vicon ID on native mocap clock ---
    id_cols, id_data = read_sto(paths['id'])
    t = id_data[:, id_cols.index('time')].astype(np.float64)
    fs = infer_fs(t)
    id_nm = id_data[:, id_cols.index(MOMENT_COL)].astype(np.float64)
    id_nmpkg = causal_butter(id_nm / mass, fs)

    # --- Vicon IK → TCN on same mocap clock (IK time → ID time) ---
    cols, ik = read_sto(paths['ik'])
    t_ik = ik[:, cols.index('time')].astype(np.float64)
    ik_r = np.deg2rad(ik[:, cols.index(JOINT_R)].astype(np.float64))
    ik_l = np.deg2rad(ik[:, cols.index(JOINT_L)].astype(np.float64))
    vel_r_ik = spline_vel(t_ik, ik_r)
    vel_l_ik = spline_vel(t_ik, ik_l)

    ang_r = fill_nan(np.interp(t, t_ik, ik_r, left=np.nan, right=np.nan))
    ang_l = fill_nan(np.interp(t, t_ik, ik_l, left=np.nan, right=np.nan))
    vel_r = fill_nan(np.interp(t, t_ik, vel_r_ik, left=np.nan, right=np.nan))
    vel_l = fill_nan(np.interp(t, t_ik, vel_l_ik, left=np.nan, right=np.nan))
    ang_r = apply_cascade_lpf(ang_r, fs, ANGLE_LPF_HZ)
    ang_l = apply_cascade_lpf(ang_l, fs, ANGLE_LPF_HZ)
    vel_r = apply_cascade_lpf(vel_r, fs, VEL_LPF_HZ)
    vel_l = apply_cascade_lpf(vel_l, fs, VEL_LPF_HZ)

    pred_raw = run_bilateral(model, ang_r, vel_r, ang_l, vel_l, window)
    pred = zp_lpf(np.nan_to_num(pred_raw, nan=0.0), fs)
    pred[~np.isfinite(pred_raw)] = np.nan

    # --- 1) Xcorr-align pred to Vicon ID/mass ---
    pred_lag, pred_xcorr = xcorr_lag_samples(pred, id_nmpkg, fs)
    pred_aligned = shift_samples(pred, pred_lag)

    # --- 2) Separately sync applied torque onto ID clock via GPIO offset ---
    d = np.load(str(paths['npz']), allow_pickle=True)
    applied = None
    applied_key = None
    for k in ('applied_torque_R', 'applied_R', 'cmd_R'):
        if k in d.files:
            applied = np.asarray(d[k], dtype=np.float64)
            applied_key = k
            break
    if applied is None:
        raise KeyError(f'No applied torque in {paths["npz"]}')
    t_telem = np.asarray(d['time'], dtype=np.float64)
    # GPIO: t_mocap ≈ t_telem + offset_s  => query ID times on telem clock as t - offset
    applied_on_id = fill_nan(np.interp(
        t, t_telem + float(gpio_offset_s), applied, left=np.nan, right=np.nan,
    ))
    applied_nmpkg = causal_butter(applied_on_id / mass, fs)

    # --- 3) GT = (ID + applied)/mass after separate syncs ---
    gt_nmpkg = causal_butter((id_nm + applied_on_id) / mass, fs)

    t_rel = t - np.nanmin(t)
    trim = (t_rel >= TRIM_START_SEC) & (t_rel <= np.nanmax(t_rel) - TRIM_END_SEC)

    rmse_id, r2_id = rmse_r2(id_nmpkg[trim], pred_aligned[trim])
    rmse_gt, r2_gt = rmse_r2(gt_nmpkg[trim], pred_aligned[trim])

    cond, speed = trial_cond_speed(stem)
    row = {
        'trial': stem,
        'subject': SUBJECT_TOKEN_TO_DIR[subject_token(stem)],
        'task': cond,
        'condition': f'{cond}_{speed}',
        'gpio_offset_s': float(gpio_offset_s),
        'pred_id_xcorr_lag_samples': int(pred_lag),
        'pred_id_xcorr_score': float(pred_xcorr),
        'applied_key': applied_key,
        'rmse_vs_id_nmpkg': rmse_id,
        'r2_vs_id': r2_id,
        'rmse_vs_id_plus_applied': rmse_gt,
        'r2_vs_id_plus_applied': r2_gt,
    }
    WAVES[stem] = {
        't': t, 'trim': trim,
        'pred': pred_aligned,
        'pred_raw_unaligned': pred,
        'id_nmpkg': id_nmpkg,
        'applied_nmpkg': applied_nmpkg,
        'gt_nmpkg': gt_nmpkg,
        'row': row,
    }
    return row


def run_all(stems: List[str]) -> pd.DataFrame:
    ckpt = torch.load(str(HIP_CKPT), map_location=DEVICE, weights_only=False)
    model = TCN(**_tcn_kwargs(ckpt['model_config']))
    model.load_state_dict(ckpt['model_state_dict'])
    model.to(DEVICE).eval()
    window = int(ckpt.get('window_size', 100))
    gpio = load_gpio_offsets()
    WAVES.clear()
    rows = []
    print(f'Checkpoint: {HIP_CKPT}')
    print(
        f'Align: xcorr pred↔ID/mass (max ±{XCORR_MAX_LAG_SEC:g}s) | '
        f'applied↔ID via GPIO offset | cascade in {ANGLE_LPF_HZ:g}/{VEL_LPF_HZ:g} Hz'
    )
    for stem in stems:
        if stem not in gpio:
            raise KeyError(f'No GPIO offset in cache for {stem}')
        row = replay_one(stem, gpio[stem], model, window)
        rows.append(row)
        print(
            f"  {stem}: pred↔ID lag={row['pred_id_xcorr_lag_samples']:+d} "
            f"(score={row['pred_id_xcorr_score']:.3f}) | "
            f"vs (ID+app) RMSE={row['rmse_vs_id_plus_applied']:.3f} R²={row['r2_vs_id_plus_applied']:.3f} | "
            f"vs ID RMSE={row['rmse_vs_id_nmpkg']:.3f} R²={row['r2_vs_id']:.3f}"
        )
    return pd.DataFrame(rows)


deploy = pd.read_csv(DEPLOY_METRICS_CSV)
stems = deploy['trial'].tolist()
print(f'Trials: {len(stems)}')

if (not FORCE_REPLAY) and METRICS_CSV.is_file():
    print(f'Loading {METRICS_CSV}')
    metrics_df = pd.read_csv(METRICS_CSV)
else:
    metrics_df = run_all(stems)
    METRICS_CSV.parent.mkdir(parents=True, exist_ok=True)
    metrics_df.to_csv(METRICS_CSV, index=False)
    print(f'Wrote {METRICS_CSV}')

print('\n=== Overall (xcorr-aligned Vicon IK→TCN) ===')
print(
    f"vs (ID+applied)/mass: RMSE {metrics_df['rmse_vs_id_plus_applied'].mean():.3f} ± "
    f"{metrics_df['rmse_vs_id_plus_applied'].std(ddof=1):.3f} | "
    f"R² {metrics_df['r2_vs_id_plus_applied'].mean():.3f} ± "
    f"{metrics_df['r2_vs_id_plus_applied'].std(ddof=1):.3f}"
)
print(
    f"vs ID/mass (after same pred xcorr): RMSE {metrics_df['rmse_vs_id_nmpkg'].mean():.3f} ± "
    f"{metrics_df['rmse_vs_id_nmpkg'].std(ddof=1):.3f} | "
    f"R² {metrics_df['r2_vs_id'].mean():.3f} ± {metrics_df['r2_vs_id'].std(ddof=1):.3f}"
)
print(
    f"pred↔ID xcorr lag: mean {metrics_df['pred_id_xcorr_lag_samples'].mean():+.1f} samples "
    f"(±{metrics_df['pred_id_xcorr_lag_samples'].std(ddof=1):.1f})"
)
metrics_df.head()




In [ ]:
FIG_DIR.mkdir(parents=True, exist_ok=True)


def _ensure(stems: List[str]):
    missing = [s for s in stems if s not in WAVES]
    if not missing:
        return
    gpio = load_gpio_offsets()
    ckpt = torch.load(str(HIP_CKPT), map_location=DEVICE, weights_only=False)
    model = TCN(**_tcn_kwargs(ckpt['model_config']))
    model.load_state_dict(ckpt['model_state_dict'])
    model.to(DEVICE).eval()
    window = int(ckpt.get('window_size', 100))
    for stem in missing:
        replay_one(stem, gpio[stem], model, window)


def plot_trial(stem: str, t_window=(20.0, 35.0)):
    _ensure([stem])
    w = WAVES[stem]
    t_rel = w['t'] - np.nanmin(w['t'])
    m = (t_rel >= t_window[0]) & (t_rel <= t_window[1]) & w['trim']
    r = w['row']
    fig, ax = plt.subplots(figsize=(10, 3.6))
    ax.plot(t_rel[m], w['gt_nmpkg'][m], color='#1e88e5', lw=1.8, label='(Vicon ID + applied) / mass')
    ax.plot(t_rel[m], w['id_nmpkg'][m], color='#90CAF9', lw=1.0, ls=':', label='Vicon ID / mass')
    ax.plot(
        t_rel[m], w['pred'][m], color='#2e7d32', lw=1.6,
        label=(
            f"Vicon IK→TCN (xcorr lag={r['pred_id_xcorr_lag_samples']:+d})  "
            f"RMSE={r['rmse_vs_id_plus_applied']:.3f} R²={r['r2_vs_id_plus_applied']:.3f}"
        ),
    )
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Hip R moment (N·m/kg)')
    ax.set_title(stem)
    ax.axhline(0, color='0.7', lw=0.6)
    ax.legend(fontsize=8, loc='upper right')
    fig.tight_layout()
    out = FIG_DIR / f'{stem}_vs_id_plus_applied.png'
    fig.savefig(out, dpi=150)
    plt.show()
    print(f'Wrote {out}')


ranked = metrics_df.sort_values('r2_vs_id_plus_applied')
exemplars = [ranked.iloc[-1]['trial'], ranked.iloc[len(ranked) // 2]['trial'], ranked.iloc[0]['trial']]
print('Exemplars vs (ID+applied) (best / mid / worst R²):', exemplars)
for stem in exemplars:
    plot_trial(stem)



## Alignment details

1. **Pred ↔ ID xcorr**: z-scored lag search of zero-phase-filtered Vicon IK→TCN vs causal ID/mass on the mocap clock. Positive lag delays the prediction.
2. **Applied ↔ ID**: telemetry `applied_R` interpolated onto ID time with the **GPIO offset** from `compare_processed_hip_exo_id` (`t_id ≈ t_telem + offset_s`). Independent of the pred xcorr.
3. **Metric**: RMSE / CoD R² of aligned pred vs causal `(ID + applied) / mass`, 10 s trim.
